In [ ]:
import os
import ast
import copy
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, top_k_accuracy_score

In [ ]:
SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
drive_root = "/content/drive/MyDrive"

matches = []

for root, dirs, files in os.walk(drive_root):
    for f in files:
        if f in [
            "scenario23_img_beam.csv",
            "scenario23_pos_beam.csv",
            "scenario23.csv"
        ]:
            matches.append(os.path.join(root, f))

print("Found full CSV files:")
for m in matches:
    print(m)

In [ ]:
FULL_CSV_PATH = "PASTE_FULL_CSV_PATH_HERE"

full_df = pd.read_csv(FULL_CSV_PATH)

print("Shape:", full_df.shape)
display(full_df.head())
print("Columns:", full_df.columns.tolist())

In [ ]:
required_cols = ["index", "unit2_pos", "unit1_beam"]

for col in required_cols:
    assert col in full_df.columns, f"Missing column: {col}"

full_df = full_df.sort_values("index").reset_index(drop=True)

print(full_df[required_cols].head())
print(full_df[required_cols].tail())

print("Total samples:", len(full_df))
print("Index min:", full_df["index"].min())
print("Index max:", full_df["index"].max())

In [ ]:
idx = full_df["index"].astype(int).values
diffs = np.diff(idx)

unique_diffs, counts = np.unique(diffs, return_counts=True)

print("Unique index differences:", unique_diffs, counts)
print("Continuous +1 transitions:", np.sum(diffs == 1))
print("Broken transitions:", np.sum(diffs != 1))

position feature function

In [ ]:
def parse_unit2_pos(pos_value):
    if isinstance(pos_value, str):
        return ast.literal_eval(pos_value)
    return pos_value


def add_position_features(df):
    df = df.copy()

    parsed = df["unit2_pos"].apply(parse_unit2_pos)

    df["pos_x"] = parsed.apply(lambda v: float(v[0]))
    df["pos_y"] = parsed.apply(lambda v: float(v[1]))

    eps = 1e-8

    df["distance"] = np.sqrt(df["pos_x"]**2 + df["pos_y"]**2)
    df["distance2"] = df["distance"] ** 2
    df["distance3"] = df["distance"] ** 3

    df["angle"] = np.arctan2(df["pos_y"], df["pos_x"])
    df["sin_angle"] = np.sin(df["angle"])
    df["cos_angle"] = np.cos(df["angle"])

    df["pos_x2"] = df["pos_x"] ** 2
    df["pos_y2"] = df["pos_y"] ** 2
    df["pos_x3"] = df["pos_x"] ** 3
    df["pos_y3"] = df["pos_y"] ** 3

    df["pos_xy"] = df["pos_x"] * df["pos_y"]

    df["unit_x"] = df["pos_x"] / (df["distance"] + eps)
    df["unit_y"] = df["pos_y"] / (df["distance"] + eps)

    df["sin2_angle"] = np.sin(2 * df["angle"])
    df["cos2_angle"] = np.cos(2 * df["angle"])
    df["sin3_angle"] = np.sin(3 * df["angle"])
    df["cos3_angle"] = np.cos(3 * df["angle"])

    df["dist_sin"] = df["distance"] * df["sin_angle"]
    df["dist_cos"] = df["distance"] * df["cos_angle"]

    return df

In [ ]:
full_df_fe = add_position_features(full_df)

feature_cols = [
    "pos_x", "pos_y",
    "distance", "distance2", "distance3",
    "angle",
    "sin_angle", "cos_angle",
    "pos_x2", "pos_y2",
    "pos_x3", "pos_y3",
    "pos_xy",
    "unit_x", "unit_y",
    "sin2_angle", "cos2_angle",
    "sin3_angle", "cos3_angle",
    "dist_sin", "dist_cos"
]

label_col = "unit1_beam"

print("Feature count:", len(feature_cols))
print("Label unique:", full_df_fe[label_col].nunique())
print("Label min/max:", full_df_fe[label_col].min(), full_df_fe[label_col].max())

display(full_df_fe[["index", "unit2_pos", *feature_cols, label_col]].head())

In [ ]:
all_labels = sorted(full_df_fe[label_col].astype(int).unique())

label_to_id = {label: i for i, label in enumerate(all_labels)}
id_to_label = {i: label for label, i in label_to_id.items()}

num_classes = len(all_labels)

print("num_classes:", num_classes)
print("label_to_id:", label_to_id)

Paper style sequence build 

In [ ]:
def build_strict_sequences_from_full_df(df, feature_cols, label_col, label_to_id, seq_len=4):
    df = df.sort_values("index").reset_index(drop=True).copy()

    idx_values = df["index"].astype(int).values
    X_raw = df[feature_cols].astype(float).values.astype(np.float32)
    y_raw = df[label_col].astype(int).values

    X_seq = []
    y_seq = []
    target_indices = []

    for i in range(seq_len - 1, len(df) - 1):
        past_idx = idx_values[i - seq_len + 1 : i + 1]
        target_idx = idx_values[i + 1]

        all_idx = np.concatenate([past_idx, [target_idx]])
        expected_idx = np.arange(all_idx[0], all_idx[0] + seq_len + 1)

        if np.array_equal(all_idx, expected_idx):
            X_seq.append(X_raw[i - seq_len + 1 : i + 1])
            y_seq.append(label_to_id[int(y_raw[i + 1])])
            target_indices.append(target_idx)

    X_seq = np.array(X_seq, dtype=np.float32)
    y_seq = np.array(y_seq, dtype=np.int64)
    target_indices = np.array(target_indices, dtype=np.int64)

    return X_seq, y_seq, target_indices


SEQ_LEN = 4

X_seq_all, y_seq_all, target_indices_all = build_strict_sequences_from_full_df(
    full_df_fe,
    feature_cols,
    label_col,
    label_to_id,
    seq_len=SEQ_LEN
)

print("X_seq_all:", X_seq_all.shape)
print("y_seq_all:", y_seq_all.shape)
print("target_indices_all:", target_indices_all.shape)

print("First target indices:", target_indices_all[:10])
print("Last target indices:", target_indices_all[-10:])

stratified split 

In [ ]:
indices = np.arange(len(X_seq_all))

idx_train, idx_temp, y_train_temp, y_temp = train_test_split(
    indices,
    y_seq_all,
    test_size=0.30,
    random_state=SEED,
    stratify=y_seq_all
)

idx_val, idx_test, y_val_temp, y_test_temp = train_test_split(
    idx_temp,
    y_temp,
    test_size=1/3,
    random_state=SEED,
    stratify=y_temp
)

X_train_seq_raw = X_seq_all[idx_train]
y_train_seq_raw = y_seq_all[idx_train]

X_val_seq_raw = X_seq_all[idx_val]
y_val_seq_raw = y_seq_all[idx_val]

X_test_seq_raw = X_seq_all[idx_test]
y_test_seq_raw = y_seq_all[idx_test]

print("Train:", X_train_seq_raw.shape, y_train_seq_raw.shape)
print("Val  :", X_val_seq_raw.shape, y_val_seq_raw.shape)
print("Test :", X_test_seq_raw.shape, y_test_seq_raw.shape)

print("Train classes:", len(np.unique(y_train_seq_raw)))
print("Val classes  :", len(np.unique(y_val_seq_raw)))
print("Test classes :", len(np.unique(y_test_seq_raw)))

In [ ]:
def show_label_distribution(name, y):
    unique, counts = np.unique(y, return_counts=True)

    df = pd.DataFrame({
        "class_id": unique,
        "count": counts,
        "percent": 100 * counts / counts.sum()
    })

    print("\n" + "=" * 60)
    print(name)
    print("samples:", len(y))
    print("unique classes:", len(unique))
    display(df.sort_values("count", ascending=False).head(20))

    return set(unique)


train_classes = show_label_distribution("TRAIN", y_train_seq_raw)
val_classes   = show_label_distribution("VAL", y_val_seq_raw)
test_classes  = show_label_distribution("TEST", y_test_seq_raw)

print("Test classes missing from train:", sorted(test_classes - train_classes))
print("Val classes missing from train :", sorted(val_classes - train_classes))

train only normalization

In [ ]:
train_flat = X_train_seq_raw.reshape(-1, X_train_seq_raw.shape[-1])

seq_mean = train_flat.mean(axis=0)
seq_std = train_flat.std(axis=0)
seq_std[seq_std == 0] = 1.0

X_train_seq = (X_train_seq_raw - seq_mean) / seq_std
X_val_seq   = (X_val_seq_raw - seq_mean) / seq_std
X_test_seq  = (X_test_seq_raw - seq_mean) / seq_std

X_train_seq = torch.tensor(X_train_seq, dtype=torch.float32)
X_val_seq   = torch.tensor(X_val_seq, dtype=torch.float32)
X_test_seq  = torch.tensor(X_test_seq, dtype=torch.float32)

y_train_seq = torch.tensor(y_train_seq_raw, dtype=torch.long)
y_val_seq   = torch.tensor(y_val_seq_raw, dtype=torch.long)
y_test_seq  = torch.tensor(y_test_seq_raw, dtype=torch.long)

print("X_train_seq:", X_train_seq.shape)
print("X_val_seq  :", X_val_seq.shape)
print("X_test_seq :", X_test_seq.shape)

print("y_train_seq:", y_train_seq.shape)
print("y_val_seq  :", y_val_seq.shape)
print("y_test_seq :", y_test_seq.shape)

assert torch.isfinite(X_train_seq).all()
assert torch.isfinite(X_val_seq).all()
assert torch.isfinite(X_test_seq).all()

dataset and dataloader 

In [ ]:
class SequenceBeamDataset(Dataset):
    def __init__(self, X_seq, y_seq):
        self.X_seq = X_seq
        self.y_seq = y_seq

    def __len__(self):
        return len(self.X_seq)

    def __getitem__(self, idx):
        return self.X_seq[idx], self.y_seq[idx]

In [ ]:
BATCH_SIZE = 128

train_dataset = SequenceBeamDataset(X_train_seq, y_train_seq)
val_dataset   = SequenceBeamDataset(X_val_seq, y_val_seq)
test_dataset  = SequenceBeamDataset(X_test_seq, y_test_seq)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

x_batch, y_batch = next(iter(train_loader))

print("x_batch:", x_batch.shape)
print("y_batch:", y_batch.shape)

seq_len = x_batch.shape[1]
input_dim = x_batch.shape[2]

print("seq_len:", seq_len)
print("input_dim:", input_dim)
print("num_classes:", num_classes)

evaluation functions 

top k evaluations 

In [ ]:
def evaluate_topk_logits(logits, labels, ks=(1, 2, 3, 5)):
    max_k = max(ks)

    _, pred = torch.topk(logits, k=max_k, dim=1)
    pred = pred.t()

    results = {}
    total = labels.size(0)

    for k in ks:
        correct = pred[:k].eq(labels.view(1, -1)).sum().item()
        results[f"top{k}"] = 100.0 * correct / total

    return results


def evaluate_model(model, loader, device, ks=(1, 2, 3, 5)):
    model.eval()

    total = 0
    total_loss = 0.0
    correct = {k: 0 for k in ks}

    criterion = nn.CrossEntropyLoss()

    with torch.no_grad():
        for x, labels in loader:
            x = x.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            logits = model(x)
            loss = criterion(logits, labels)

            bs = labels.size(0)
            total += bs
            total_loss += loss.item() * bs

            max_k = max(ks)
            _, pred = torch.topk(logits, k=max_k, dim=1)
            pred = pred.t()

            for k in ks:
                correct[k] += pred[:k].eq(labels.view(1, -1)).sum().item()

    metrics = {"loss": total_loss / total}

    for k in ks:
        metrics[f"top{k}"] = 100.0 * correct[k] / total

    return metrics

mlp baseline 

In [ ]:
class SequenceMLP(nn.Module):
    def __init__(
        self,
        seq_len,
        input_dim,
        num_classes,
        hidden_dims=(256, 128, 64),
        dropout=0.35
    ):
        super().__init__()

        flat_dim = seq_len * input_dim

        h1, h2, h3 = hidden_dims

        self.net = nn.Sequential(
            nn.Linear(flat_dim, h1),
            nn.BatchNorm1d(h1),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(h1, h2),
            nn.BatchNorm1d(h2),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(h2, h3),
            nn.BatchNorm1d(h3),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(h3, num_classes)
        )

    def forward(self, x):
        x = x.reshape(x.size(0), -1)
        return self.net(x)

lstm baseline 

In [ ]:
class SequenceLSTM(nn.Module):
    def __init__(
        self,
        input_dim,
        num_classes,
        embed_dim=64,
        hidden_dim=96,
        num_layers=1,
        dropout=0.35
    ):
        super().__init__()

        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.0
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 96),
            nn.LayerNorm(96),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(96, num_classes)
        )

    def forward(self, x):
        x = self.input_proj(x)
        out, _ = self.lstm(x)
        last = out[:, -1, :]
        return self.classifier(last)

trainer with anti overfit control 

In [ ]:
def train_with_early_stopping(
    model,
    train_loader,
    val_loader,
    device,
    epochs=100,
    lr=1e-4,
    weight_decay=1e-3,
    label_smoothing=0.05,
    patience=12,
    min_delta=1e-4,
    save_path="best_model.pth"
):
    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

    optimizer = optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=5
    )

    best_val_loss = float("inf")
    best_val_top1 = -1
    patience_counter = 0

    history = []

    for epoch in range(1, epochs + 1):
        model.train()

        train_total = 0
        train_loss_sum = 0.0
        train_correct = 0

        for x, labels in train_loader:
            x = x.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad()

            logits = model(x)
            loss = criterion(logits, labels)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()

            bs = labels.size(0)
            train_total += bs
            train_loss_sum += loss.item() * bs
            train_correct += (logits.argmax(dim=1) == labels).sum().item()

        train_loss = train_loss_sum / train_total
        train_acc = 100.0 * train_correct / train_total

        val_metrics = evaluate_model(model, val_loader, device, ks=(1, 2, 3, 5))
        val_loss = val_metrics["loss"]
        val_top1 = val_metrics["top1"]

        scheduler.step(val_loss)

        current_lr = optimizer.param_groups[0]["lr"]

        row = {
            "epoch": epoch,
            "lr": current_lr,
            "train_loss": train_loss,
            "train_top1": train_acc,
            "val_loss": val_loss,
            **{f"val_{k}": v for k, v in val_metrics.items() if k != "loss"}
        }

        history.append(row)

        print(
            f"Epoch {epoch:03d} | "
            f"LR {current_lr:.2e} | "
            f"Train Loss {train_loss:.4f} | "
            f"Train Top1 {train_acc:.2f} | "
            f"Val Loss {val_loss:.4f} | "
            f"Val Top1 {val_metrics['top1']:.2f} | "
            f"Top2 {val_metrics['top2']:.2f} | "
            f"Top3 {val_metrics['top3']:.2f} | "
            f"Top5 {val_metrics['top5']:.2f}"
        )

        improved = val_loss < (best_val_loss - min_delta)

        if improved:
            best_val_loss = val_loss
            best_val_top1 = val_top1
            patience_counter = 0
            torch.save(copy.deepcopy(model.state_dict()), save_path)
            print("Saved best model")
        else:
            patience_counter += 1
            print(f"Patience: {patience_counter}/{patience}")

        if patience_counter >= patience:
            print("Early stopping triggered")
            break

    print("Best Val Loss:", best_val_loss)
    print("Best Val Top1:", best_val_top1)

    return pd.DataFrame(history)

train mlp 

In [ ]:
set_seed(SEED)

mlp_model = SequenceMLP(
    seq_len=seq_len,
    input_dim=input_dim,
    num_classes=num_classes,
    hidden_dims=(256, 128, 64),
    dropout=0.35
).to(device)

mlp_save_path = "/content/drive/MyDrive/best_sequence_mlp_clean.pth"

history_mlp = train_with_early_stopping(
    model=mlp_model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    epochs=100,
    lr=1e-4,
    weight_decay=1e-3,
    label_smoothing=0.05,
    patience=12,
    save_path=mlp_save_path
)

test mlp 

In [ ]:
mlp_eval = SequenceMLP(
    seq_len=seq_len,
    input_dim=input_dim,
    num_classes=num_classes,
    hidden_dims=(256, 128, 64),
    dropout=0.35
).to(device)

mlp_eval.load_state_dict(torch.load(mlp_save_path, map_location=device))

mlp_test_metrics = evaluate_model(
    mlp_eval,
    test_loader,
    device,
    ks=(1, 2, 3, 5)
)

print("Clean Sequence MLP Test Metrics:")
print(mlp_test_metrics)

lstm model 

train lstm 

In [ ]:
set_seed(SEED)

lstm_model = SequenceLSTM(
    input_dim=input_dim,
    num_classes=num_classes,
    embed_dim=64,
    hidden_dim=96,
    num_layers=1,
    dropout=0.35
).to(device)

lstm_save_path = "/content/drive/MyDrive/best_sequence_lstm_clean.pth"

history_lstm = train_with_early_stopping(
    model=lstm_model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    epochs=100,
    lr=1e-4,
    weight_decay=1e-3,
    label_smoothing=0.05,
    patience=12,
    save_path=lstm_save_path
)

test lstm 